In [ ]:
import os
import csv
import ast
import pickle
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import matplotlib as mpl

import sys
sys.path.insert(0, str(Path("..").resolve()))
from utils import savefig

plt.rcParams['font.size'] = 14

mpl.rcParams['svg.fonttype'] = 'none'

In [ ]:
colormap = {
    "forward_asymmetry": "tab:purple",
    "temporal_factor": "tab:blue",
    "explained_variance": "tab:green",
    "cross_decoding": "tab:orange",
    "accuracy": "black"
}

gamma_names = ["0", "02", "04", "06", "08", "10"]
temporal_discount_factors = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
noise_levels = [0, 0.2, 0.4, 0.6, 0.8, 1]
noise_names = ["0", "02", "04", "06", "08", "1"]

In [ ]:
df = pd.read_pickle("../data/df_models.pkl")
df_reservoir = pd.read_pickle("../data/df_reservoir.pkl")
df_tcm = pd.read_pickle("../data/df_tcm.pkl")
df_perf = pd.read_pickle("../data/df_perf.pkl")

### performance

In [ ]:
# training_accuracy_by_tdf = df_tdf[(df_tdf["pretrained"] == True) & (df_tdf["eta"] == 0.01)].groupby("temporal_discount_factor")["training_accuracy"].mean()
fig_num = ["A", "B", "C", "D", "E", "F", "G", "H", "I", "J", "K", "L", "M", "N", "O", "P", "Q", "R", "S", "T", "U", "V", "W", "X", "Y", "Z"]
for n, noise_level in enumerate(noise_levels):
    df_filtered = df[(df["accuracy"] > 0.6) & (df["noise_level"] == noise_level)].copy()
    training_accuracy_by_tdf = df_filtered.groupby("temporal_discount_factor")["training_accuracy"].mean()

    colors = plt.cm.viridis(np.linspace(0, 1, len(training_accuracy_by_tdf)))
    if n == 5:
        plt.figure(figsize=(5.2, 3.3), dpi=180)
    else:
        plt.figure(figsize=(3.5, 3.3), dpi=180)
    for i, (tdf, acc) in enumerate(training_accuracy_by_tdf.items()):
        plt.plot(acc[:200], label=f"{tdf:.1f}", color=colors[i])
        std_dev = np.std(np.stack(df_filtered[df_filtered["temporal_discount_factor"] == tdf]["training_accuracy"]), axis=0)
        plt.fill_between(range(len(acc[:200])), acc[:200] - std_dev, acc[:200] + std_dev, color=colors[i], alpha=0.2)
        
    # set the legend to the right of the plot
    plt.title(f"% WM flushed = {noise_level:.1f}", fontsize=14)
    ax = plt.gca()
    if noise_level == noise_levels[-1]:
        ax.legend(loc="center left", bbox_to_anchor=(1, 0.5), frameon=False, title="discount factor")
    plt.xlabel("Training epochs (1000)")
    plt.ylabel("Training accuracy")
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    plt.tight_layout()
    savefig("../figures/supp1", "suppfig1"+fig_num[n], format="svg", close=False)
    plt.show()


In [ ]:

fig_num = ["G", "H", "I", "J", "K", "L", "M", "N", "O", "P", "Q", "R", "S", "T", "U", "V", "W", "X", "Y", "Z"]
for n, tdf in enumerate(temporal_discount_factors):
    df_filtered = df[(df["accuracy"] > 0.6) & (df["temporal_discount_factor"] == tdf)].copy()
    training_accuracy_by_tdf = df_filtered.groupby("noise_level")["training_accuracy"].mean()

    colors = plt.cm.viridis(np.linspace(0, 1, len(training_accuracy_by_tdf)))
    if n == 5:
        plt.figure(figsize=(5.2, 3.3), dpi=180)
    else:
        plt.figure(figsize=(3.5, 3.3), dpi=180)
    for i, (noise_level, acc) in enumerate(training_accuracy_by_tdf.items()):
        plt.plot(acc[:200], label=f"{noise_level:.1f}", color=colors[i])
        std_dev = np.std(np.stack(df_filtered[df_filtered["noise_level"] == noise_level]["training_accuracy"]), axis=0)
        plt.fill_between(range(len(acc[:200])), acc[:200] - std_dev, acc[:200] + std_dev, color=colors[i], alpha=0.2)
        
    # set the legend to the right of the plot
    plt.title(f"Discount factor = {tdf:.1f}", fontsize=14)
    ax = plt.gca()
    if tdf == temporal_discount_factors[-1]:
        ax.legend(loc="center left", bbox_to_anchor=(1, 0.5), frameon=False, title="% WM flushed")
    plt.xlabel("Training epochs (1000)")
    plt.ylabel("Training accuracy")
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    plt.tight_layout()
    savefig("../figures/supp1", "suppfig1"+fig_num[n], format="svg", close=False)
    plt.show()


### metrics changing with hyperparams

In [ ]:

def plot_mean_std_scatter(x, y, yerr, x_scatter, y_scatter, x_label, y_label, color='tab:blue', figname=None, vary_x=0.15):
    fig = plt.figure(figsize=(3.4, 3.3), dpi=180)
    plt.errorbar(np.arange(len(y))-vary_x, y, yerr=yerr, fmt='o', alpha=0.8, capsize=3, color=color)
    unique_x_values = np.unique(x_scatter)
    
    # Update the scatter plot to use integer x-axis
    x_scatter_int = np.array([np.where(unique_x_values == x)[0][0] for x in x_scatter])
    plt.scatter(x_scatter_int + vary_x, y_scatter, alpha=0.3, color=color, s=15)
    plt.xticks(ticks=range(len(unique_x_values)), labels=[f"{x_val}" for x_val in unique_x_values])
    plt.xlabel(x_label)
    plt.ylabel(y_label)
    ax = plt.gca()
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    if figname:
        savefig("../figures/supp2", figname, format="svg", close=False)
    plt.show()

In [ ]:
df_tdf_filtered = df[(df["accuracy"] > 0.6) & (df["noise_level"] == 1.0)]
print(len(df_tdf_filtered))

In [ ]:
accuracy_mean_by_tdf = df_tdf_filtered.groupby("temporal_discount_factor")["accuracy"].mean()
accuracy_std_by_tdf = df_tdf_filtered.groupby("temporal_discount_factor")["accuracy"].std()

temporal_factor_mean_by_tdf = df_tdf_filtered.groupby("temporal_discount_factor")["temporal_factor"].mean()
temporal_factor_std_by_tdf = df_tdf_filtered.groupby("temporal_discount_factor")["temporal_factor"].std()

forward_asymmetry_mean_by_tdf = df_tdf_filtered.groupby("temporal_discount_factor")["forward_asymmetry"].mean()
forward_asymmetry_std_by_tdf = df_tdf_filtered.groupby("temporal_discount_factor")["forward_asymmetry"].std()

tdf_all = df_tdf_filtered["temporal_discount_factor"]
accuracy_all = df_tdf_filtered["accuracy"]
forward_asymmetry_all = df_tdf_filtered["forward_asymmetry"]
temporal_factor_all = df_tdf_filtered["temporal_factor"]
temporal_discount_factors = df_tdf_filtered["temporal_discount_factor"].unique()


plot_mean_std_scatter(temporal_discount_factors, accuracy_mean_by_tdf, accuracy_std_by_tdf, tdf_all, accuracy_all, "Discount factor", "Task accuracy", color=colormap["accuracy"])
plot_mean_std_scatter(temporal_discount_factors, temporal_factor_mean_by_tdf, temporal_factor_std_by_tdf, tdf_all, temporal_factor_all, "Discount factor", 
                    "Temporal organization\nscore", color=colormap["temporal_factor"], figname="suppfig2B")
plot_mean_std_scatter(temporal_discount_factors, forward_asymmetry_mean_by_tdf, forward_asymmetry_std_by_tdf, tdf_all, forward_asymmetry_all, "Discount factor", 
                    "Forward asymmetry", color=colormap["forward_asymmetry"], figname="suppfig2A")



# change with tdf
tdf_all = df_tdf_filtered["temporal_discount_factor"]
accuracy_all = df_tdf_filtered["accuracy"]
forward_asymmetry_all = df_tdf_filtered["forward_asymmetry"]
temporal_factor_all = df_tdf_filtered["temporal_factor"]
variance_explained_index_all = df_tdf_filtered["explained_variance_index"]
variance_explained_identity_all = df_tdf_filtered["explained_variance_identity"]
cross_decoding_accuracy_index_all = df_tdf_filtered["cross_decoding_accuracy_index"]
cross_decoding_accuracy_identity_all = df_tdf_filtered["cross_decoding_accuracy_identity"]

variance_explained_index_mean_by_tdf = df_tdf_filtered.groupby("temporal_discount_factor")["explained_variance_index"].mean()
variance_explained_identity_mean_by_tdf = df_tdf_filtered.groupby("temporal_discount_factor")["explained_variance_identity"].mean()
cross_decoding_accuracy_index_mean_by_tdf = df_tdf_filtered.groupby("temporal_discount_factor")["cross_decoding_accuracy_index"].mean()
cross_decoding_accuracy_identity_mean_by_tdf = df_tdf_filtered.groupby("temporal_discount_factor")["cross_decoding_accuracy_identity"].mean()

variance_explained_index_std_by_tdf = df_tdf_filtered.groupby("temporal_discount_factor")["explained_variance_index"].std()
variance_explained_identity_std_by_tdf = df_tdf_filtered.groupby("temporal_discount_factor")["explained_variance_identity"].std()
cross_decoding_accuracy_index_std_by_tdf = df_tdf_filtered.groupby("temporal_discount_factor")["cross_decoding_accuracy_index"].std()
cross_decoding_accuracy_identity_std_by_tdf = df_tdf_filtered.groupby("temporal_discount_factor")["cross_decoding_accuracy_identity"].std()


plot_mean_std_scatter(temporal_discount_factors, variance_explained_index_mean_by_tdf, variance_explained_index_std_by_tdf, 
                      tdf_all, variance_explained_index_all, "Discount factor", "Variance explained\n(index)", color=colormap["explained_variance"],
                      figname="suppfig2C")

plot_mean_std_scatter(temporal_discount_factors, variance_explained_identity_mean_by_tdf, variance_explained_identity_std_by_tdf, 
                      tdf_all, variance_explained_identity_all, "Discount factor", "Variance explained\n(identity)", color=colormap["explained_variance"],
                      figname="suppfig2D")

plot_mean_std_scatter(temporal_discount_factors, cross_decoding_accuracy_index_mean_by_tdf, cross_decoding_accuracy_index_std_by_tdf, 
                      tdf_all, cross_decoding_accuracy_index_all, "Discount factor", "Cross decoding accuracy\n(index)", color=colormap["cross_decoding"],
                      figname="suppfig2E")

plot_mean_std_scatter(temporal_discount_factors, cross_decoding_accuracy_identity_mean_by_tdf, cross_decoding_accuracy_identity_std_by_tdf, 
                      tdf_all, cross_decoding_accuracy_identity_all, "Discount factor", "Cross decoding accuracy\n(identity)", color=colormap["cross_decoding"],
                      figname="suppfig2F")


In [ ]:
df_noise_filtered = df[(df["accuracy"] > 0.6) & (df["temporal_discount_factor"] == 1.0)]
print(len(df_noise_filtered))

In [ ]:
accuracy_mean_by_noise = df_noise_filtered.groupby("noise_level")["accuracy"].mean()
accuracy_std_by_noise = df_noise_filtered.groupby("noise_level")["accuracy"].std()

temporal_factor_mean_by_noise = df_noise_filtered.groupby("noise_level")["temporal_factor"].mean()
temporal_factor_std_by_noise = df_noise_filtered.groupby("noise_level")["temporal_factor"].std()

forward_asymmetry_mean_by_noise = df_noise_filtered.groupby("noise_level")["forward_asymmetry"].mean()
forward_asymmetry_std_by_noise = df_noise_filtered.groupby("noise_level")["forward_asymmetry"].std()

noise_all = df_noise_filtered["noise_level"]
accuracy_all = df_noise_filtered["accuracy"]
forward_asymmetry_all = df_noise_filtered["forward_asymmetry"]
temporal_factor_all = df_noise_filtered["temporal_factor"]
noise_levels = df_noise_filtered["noise_level"].unique()


plot_mean_std_scatter(temporal_discount_factors, accuracy_mean_by_noise, accuracy_std_by_noise, noise_all, accuracy_all, "% WM flushed", "Task accuracy", color=colormap['accuracy'])
plot_mean_std_scatter(temporal_discount_factors, temporal_factor_mean_by_noise, temporal_factor_std_by_noise, noise_all, temporal_factor_all, "% WM flushed", 
                    "Temporal organization\nscore", color=colormap["temporal_factor"], figname="suppfig2H")
plot_mean_std_scatter(temporal_discount_factors, forward_asymmetry_mean_by_noise, forward_asymmetry_std_by_noise, noise_all, forward_asymmetry_all, "% WM flushed", 
                    "Forward asymmetry", color=colormap["forward_asymmetry"], figname="suppfig2G")



variance_explained_index_all = df_noise_filtered["explained_variance_index"]
variance_explained_identity_all = df_noise_filtered["explained_variance_identity"]
cross_decoding_accuracy_index_all = df_noise_filtered["cross_decoding_accuracy_index"]
cross_decoding_accuracy_identity_all = df_noise_filtered["cross_decoding_accuracy_identity"]

variance_explained_index_mean_by_noise = df_noise_filtered.groupby("noise_level")["explained_variance_index"].mean()
variance_explained_index_std_by_noise = df_noise_filtered.groupby("noise_level")["explained_variance_index"].std()
variance_explained_identity_mean_by_noise = df_noise_filtered.groupby("noise_level")["explained_variance_identity"].mean()
variance_explained_identity_std_by_noise = df_noise_filtered.groupby("noise_level")["explained_variance_identity"].std()

cross_decoding_accuracy_index_mean_by_noise = df_noise_filtered.groupby("noise_level")["cross_decoding_accuracy_index"].mean()
cross_decoding_accuracy_index_std_by_noise = df_noise_filtered.groupby("noise_level")["cross_decoding_accuracy_index"].std()
cross_decoding_accuracy_identity_mean_by_noise = df_noise_filtered.groupby("noise_level")["cross_decoding_accuracy_identity"].mean()
cross_decoding_accuracy_identity_std_by_noise = df_noise_filtered.groupby("noise_level")["cross_decoding_accuracy_identity"].std()


plot_mean_std_scatter(noise_levels, variance_explained_index_mean_by_noise, variance_explained_index_std_by_noise, noise_all, variance_explained_index_all, "% WM flushed", 
                    "Variance explained\n(index)", colormap["explained_variance"], figname="suppfig2I")
plot_mean_std_scatter(noise_levels, variance_explained_identity_mean_by_noise, variance_explained_identity_std_by_noise, noise_all, variance_explained_identity_all, "% WM flushed", 
                    "Variance explained\n(identity)", colormap["explained_variance"], figname="suppfig2J")
plot_mean_std_scatter(noise_levels, cross_decoding_accuracy_index_mean_by_noise, cross_decoding_accuracy_index_std_by_noise, noise_all, cross_decoding_accuracy_index_all, "% WM flushed", 
                    "Cross decoding accuracy\n(index)", colormap["cross_decoding"], figname="suppfig2K")
plot_mean_std_scatter(noise_levels, cross_decoding_accuracy_identity_mean_by_noise, cross_decoding_accuracy_identity_std_by_noise, noise_all, cross_decoding_accuracy_identity_all, "% WM flushed", 
                    "Cross decoding accuracy\n(identity)", colormap["cross_decoding"], figname="suppfig2L")
